In Python, everything is an object living in a pool of memory called the Heap. When Python reads a def statement, it does not run the code inside. Instead, it creates a Function Object in the Heap.

The name of your function is just a variable pointing to that object.

The "Default Argument" Trap (Evaluated Only Once)
When the function object is created in the Heap, Python evaluates the default arguments at that exact moment and stores them inside a special attribute on the function object called __defaults__.

In [ ]:
def add_item(item, bucket=[]):
    bucket.append(item)
    return bucket


# Let's peek into the memory of the function itself!
print(add_item.__defaults__)
# Output: ([],) -> A tuple containing one empty list.

([],)


In [3]:
b = add_item("apple")
print(b)  # Output: ['apple']

['apple']


In [6]:
add_item.__defaults__[0]

['apple']

In [7]:
id(b)

2166046943232

In [8]:
id(add_item.__defaults__[0])

2166046943232

both are pointing to same things in memory, which is the default list. This is why we see the unexpected behavior of the default list being modified across function calls.

In [ ]:
id(add_item.__defaults__[0]) == add_item("banana")  # Output: True

False

In [11]:
b

['apple', 'banana']

In [ ]:
id(add_item.__defaults__[0]) == id(add_item("banana"))

True

In [13]:
b

['apple', 'banana', 'banana']

In [14]:
add_item.__defaults__[0]

['apple', 'banana', 'banana']

In [15]:
del b

In [22]:
print(add_item("orange"))  # Output: ['apple', 'banana', 'orange']

['apple', 'banana', 'orange']


The Fix: This is why you should always use None for mutable defaults. None is immutable.

In [23]:
def add_item_safe(item, bucket=None):
    if bucket is None:
        bucket = []  # Creates a NEW list in the stack frame on every call
    bucket.append(item)
    return bucket

In [24]:
print(add_item_safe("grape"))  # Output: ['grape']
print(add_item_safe("melon"))  # Output: ['melon']

['grape']
['melon']


In [25]:
add_item_safe.__defaults__

(None,)

Phase 2: Function Calls & The Stack Frame
When you use parentheses () to call a function, Python creates a Stack Frame.

Creation: A Stack Frame is a temporary workspace in memory. It holds the local variables for that specific function call.

Execution: The code inside the function runs, using the variables in this temporary frame.

Destruction: As soon as the function hits return (or finishes its last line), the Stack Frame is destroyed. All local variables inside it are deleted (unless something else is still pointing to them).

Phase 3: Returning an Inner Function (Closures)
Things get wild when a function defines an inner function and returns it. Let's look at a classic setup:

In [26]:
def outer_func(multiplier):
    # This is a local variable in outer_func's stack frame
    message = "Multiplying by " + str(multiplier)

    def inner_func(number):
        print(message)
        return number * multiplier

    return inner_func


# Call the outer function
multiply_by_3 = outer_func(3)

In [27]:
multiply_by_3(10)  # Output: "Multiplying by 3" followed by 30

Multiplying by 3


30

In [ ]:
multiply_by_3(5)  # Output: "Multiplying by 3" followed by 15

Multiplying by 3


15

Step-by-Step Memory Breakdown of the above code:
Call outer_func(3): Python creates a Stack Frame for outer_func. It stores multiplier = 3 and message = "Multiplying by 3".

Define inner_func: Python creates a new Function Object in the Heap for inner_func.

Return inner_func: The outer function returns the memory address of inner_func to the variable multiply_by_3.

Destruction of outer_func: The Stack Frame for outer_func is destroyed.

The Big Question: If outer_func is destroyed, its variables (multiplier and message) should be gone. So how does multiply_by_3(10) still know the multiplier is 3?

The Answer: Closures (The Backpack)
When Python creates inner_func, it notices that inner_func uses variables from outer_func. Because Python knows outer_func is going to be destroyed, it takes those specific variables and packs them into a "backpack" attached to inner_func.

This backpack is called a Closure. It keeps those specific heap objects alive even after the outer stack frame is gone.

We can actually see this backpack in memory using the __closure__ attribute:

In [ ]:
# Look inside the returned function's backpack
cells = multiply_by_3.__closure__

for cell in cells:
    print(cell.cell_contents)

# Output:
# Multiplying by 3   <-- The 'message' variable survived!
# 3                  <-- The 'multiplier' variable survived!

Multiplying by 3
3


Summary of the Lifecycle
def statement: Creates a Function Object in the Heap. Default arguments are evaluated now and permanently attached to this object.

Calling a function: Creates a temporary Stack Frame. Local variables live here.

Returning: The Stack Frame is destroyed.

Inner Functions: If an inner function is returned, it creates a Closure. It grabs the specific outer variables it needs and binds them to itself in the Heap. The outer stack frame still dies, but the specific data lives on inside the inner function's __closure__ attribute.

__defaults__ saves a memory address (a pointer), NEVER the actual object.

In Python, a variable or an attribute (like __defaults__) is just a piece of paper with a memory address written on it. The "actual object" always lives somewhere else, in a giant warehouse called the Heap.

Let's break down exactly what an address is, who points to what, and why the immutable 0 seems to "reset" while the mutable list [] does not.

Part 1: What is a Memory Address?
Think of the Heap as a massive street full of houses.

The Object (like the number 0 or a list []) is the physical house.

The Memory Address is the street address (e.g., Memory Location 1000).

A Variable is just a sticky note with "Location 1000" written on it.

When we ask Python for the id(), it gives us the memory address.

Part 2: The Immutable Default (total = 0) - Step by Step
Why does total seem to reset? It is because of Reassignment. Reassignment means crossing out the old address on your sticky note and writing a new one.

Step 1: Defining the function

In [31]:
def add_score(points, total=0):
    total = total + points
    return total

When Python reads this, it creates the integer 0 in the Heap. Let's say it lives at Address 1000.
Python creates the __defaults__ tuple. Inside it, it writes down Address 1000.

Memory State right now:
__defaults__[0] ----> points to ----> Address 1000 (holds 0)

Step 2: The First Function Call -> add_score(5)
Python creates a temporary Stack Frame for this specific call. It needs a local variable named total. Because you didn't provide a second argument, it looks at __defaults__, sees Address 1000, and copies that address onto the local total sticky note.

Stack Frame State:
Local total ----> points to ----> Address 1000 (holds 0)

Step 3: The Math Happens -> total = total + points
This is the crucial moment. Python calculates 0 + 5 = 5.
Because integers are immutable, Python cannot change the 0 at Address 1000. Instead, it builds a brand new integer 5 somewhere else in the Heap—let's say Address 5555.

Because of the = sign, Python takes the local total sticky note, crosses out 1000, and writes down 5555.

Stack Frame State:
Local total ----> points to ----> Address 5555 (holds 5)

Heap State (Untouched):
__defaults__[0] ----> points to ----> Address 1000 (holds 0)

Notice that the __defaults__ sticky note did not move! It still points to Address 1000.

Step 4: The Second Function Call -> add_score(10)
The first Stack Frame was destroyed. A new one is created. Python again looks at __defaults__. What address is written there? Address 1000.
So, the new local total variable starts pointing to Address 1000 (which still holds 0). This is why it looks like it "reset". It didn't reset; it just went back to looking at the original, untouched default address.

Part 3: The Mutable Default (bucket=[]) - Step by Step
Why doesn't the list reset? Because of Mutation. Mutation means following the address to the house, going inside, and rearranging the furniture. You do not cross out the address on your sticky note.

Step 1: Defining the function

In [32]:
def add_item(item, bucket=[]):
    bucket.append(item)
    return bucket

Python creates an empty list [] in the Heap at Address 2000.
It writes Address 2000 inside __defaults__.

Memory State right now:
__defaults__[0] ----> points to ----> Address 2000 (holds [])

Step 2: The First Function Call -> add_item("Apple")
Python creates the Stack Frame. The local variable bucket copies the address from __defaults__.

Stack Frame State:
Local bucket ----> points to ----> Address 2000 (holds [])

Step 3: The Modification Happens -> bucket.append("Apple")
Notice there is no equals sign (=) here. We are not reassigning the variable.
Python follows the local bucket sticky note to Address 2000. It opens the front door of the list object and puts "Apple" inside.

Stack Frame State:
Local bucket ----> points to ----> Address 2000 (holds ["Apple"])

Heap State (Mutated):
__defaults__[0] ----> points to ----> Address 2000 (holds ["Apple"])

Because both sticky notes point to Address 2000, and Address 2000 was physically altered, __defaults__ now points to a modified list!

Step 4: The Second Function Call -> add_item("Banana")
The first Stack Frame is destroyed. A new one is created.
Python looks at __defaults__. What address is written there? Address 2000.
The new local bucket sticky note gets Address 2000 written on it.

But when Python follows Address 2000 to the Heap, the house already has "Apple" sitting inside it! So when it does append("Banana"), it adds to the existing items.

The Golden Rule Summary
To answer your question "is it diff in case for mutable and immutable?":

No, the way Python saves the default is exactly the same: it saves the memory address.

The difference is entirely about the = sign vs the .append() method inside your function block:

total = ... changes the address written on the local variable's sticky note. The original __defaults__ sticky note is left pointing to the original object.

bucket.append(...) follows the address and changes the contents of the house. Since __defaults__ still points to that exact same house, the default is permanently altered for all future function calls.

Proof 1: The Immutable Default (Integer)
Notice how the total ID starts the same as the default, but changes the moment you do math.

In [35]:
def add_score(points, total=0):
    print(f"  [Inside] Start ID of 'total': {id(total)}")
    # print(f"Default ID in Heap inside function: {id(add_score.__defaults__[0])}")
    total = total + points  # The Reassignment happens here

    print(f"  [Inside] End ID of 'total':   {id(total)} <-- It moved!")
    return total


print("--- IMMUTABLE PROOF ---")
# 1. Print the ID of the default stored in the Heap
print(f"Default ID in Heap: {id(add_score.__defaults__[0])}")

print("\nCall 1:")
add_score(5)

print("\nCall 2:")
add_score(10)

# 2. Prove the default in the Heap was never touched
print(f"\nFinal Default ID:   {id(add_score.__defaults__[0])} <-- Untouched!")

--- IMMUTABLE PROOF ---
Default ID in Heap: 140718490186840

Call 1:
  [Inside] Start ID of 'total': 140718490186840
  [Inside] End ID of 'total':   140718490187000 <-- It moved!

Call 2:
  [Inside] Start ID of 'total': 140718490186840
  [Inside] End ID of 'total':   140718490187160 <-- It moved!

Final Default ID:   140718490186840 <-- Untouched!


Proof 2: The Mutable Default (List)
Notice how the bucket ID never changes, which means both the local variable and the __defaults__ attribute are modifying the exact same object in the Heap.

In [ ]:
def add_item(item, bucket=[]):
    print(f"  [Inside] Start ID of 'bucket': {id(bucket)}")

    bucket.append(item)  # The Mutation happens here

    print(f"  [Inside] End ID of 'bucket':   {id(bucket)} <-- Did not move!")
    return bucket


print("\n--- MUTABLE PROOF ---")
# 1. Print the ID of the default list stored in the Heap
print(f"Default ID in Heap: {id(add_item.__defaults__[0])}")

print("\nCall 1:")
add_item("Apple")

print("\nCall 2:")
add_item("Banana")

# 2. Prove the default in the Heap was permanently mutated
print(f"\nFinal Default ID:   {id(add_item.__defaults__[0])} <-- Exact same ID!")
print(f"Final Default Data: {add_item.__defaults__[0]} <-- Data changed permanently!")


--- MUTABLE PROOF ---
Default ID in Heap: 2166046789632

Call 1:
  [Inside] Start ID of 'bucket': 2166046789632
  [Inside] End ID of 'bucket':   2166046789632 <-- Did not move!

Call 2:
  [Inside] Start ID of 'bucket': 2166046789632
  [Inside] End ID of 'bucket':   2166046789632 <-- Did not move!

Final Default ID:   2166046789632 <-- Exact same ID!
Final Default Data: ['Apple', 'Banana'] <-- Data changed permanently!


Wow: I can call self fns default ion definition itself:

It feels like a paradox, doesn't it? It feels like the function is trying to look at itself in the mirror before it has even finished being built.

The reason it does not throw an error comes down to when the code is actually executed versus when the function is created.

Here is the exact timeline of how Python avoids the error.

The Timeline of a Function
To Python, the code inside the function block doesn't really mean anything until you actually call the function.

Step 1: The Definition (Building the House)
When Python reads the def add_score(points, total=0): line, it does three things immediately:

It creates the Function Object in the Heap.

It evaluates the default 0 and stores it in __defaults__ on that Object.

Crucially: It creates a sticky note in the Global Scope named add_score and points it to that Function Object.

At this exact moment, Python stops. It does not run the code inside the function. It just stores the text of your code for later. The function is fully built, registered, and named in the Global Scope.

Step 2: The Call (Opening the Door)
Later in your script, you actually call the function: add_score(5).

Now, Python steps inside the function and starts running your code line by line.

Step 3: Name Resolution (Looking for the Sticky Note)
Inside the function, you wrote:
print(add_score.__defaults__)

Here is exactly what Python's brain does:

"The user wants a variable named add_score. Is there a local sticky note named add_score in this Stack Frame?" -> No.

"Okay, let's look out the window at the Global Scope. Is there a global sticky note named add_score?" -> Yes! (It was created back in Step 1).

"Great. Follow that sticky note to the Heap, and grab its __defaults__ attribute."

The Code Proof
You can see this timeline in action. If you try to access the function's name before the def block is completely finished, it crashes. But inside the block, it's safe because the block runs after the def is finished.

In [1]:
# 1. This would CRASH if uncommented, because the 'test_func'
# sticky note doesn't exist in the global scope yet.
# print(test_func.__defaults__)


def test_func(x=10):
    # 3. By the time this code actually runs, the global 'test_func'
    # sticky note has existed for a long time!
    print("Inside the function, looking at myself:", test_func.__defaults__)


# 2. Python finishes building the function here.
# The global 'test_func' sticky note now exists.

print("Outside the function:", test_func.__defaults__)

# 4. We finally call the function, triggering the code inside.
test_func()

Outside the function: (10,)
Inside the function, looking at myself: (10,)


Summary
It doesn't throw an error because the inside of a function is executed in the future. By the time the "future" arrives and you call the function, the add_score object has already been fully constructed, and its name is safely written on a global sticky note for anyone (even the function itself) to find.

To understand every single case of scoping, closures, and defaults in Python, we have to start with one universal truth about Python's architecture: Variables are not boxes that hold data; they are sticky notes (pointers) attached to objects in the Heap.

If you memorize this "sticky note" rule, every weird behavior in Python instantly makes sense.

Here is the exhaustive breakdown of every scenario involving scope, memory, mutability, and closures.

Phase 1: The Core Rule of Mutability vs. Immutability
Everything in Python lives in the Heap. But objects fall into two categories:

Immutables (Integers, Strings, Tuples): The object is locked in stone. If you try to change it, Python creates a brand new object in the Heap and moves your sticky note to it.

Mutables (Lists, Dictionaries, Sets): The object is a container. You can reach inside the container and change the contents without moving the sticky note.

In [6]:
# IMMUTABLE EXAMPLE
a = 10  # Sticky note 'a' points to integer 10
print(id(a))  # Output: 140732...
a = a + 1  # Integers are immutable! Python creates an 11, and moves 'a' to it.
print(id(a))  # Output: 140731... (Different ID. The sticky note moved.)

# MUTABLE EXAMPLE
lst = [1, 2]  # Sticky note 'lst' points to a list
print(id(lst))  # Output: 20984...
lst.append(3)  # Lists are mutable! We reached inside the existing list.
print(id(lst))  # Output: 20984... (Exact same ID. The sticky note didn't move.)

140718490187160
140718490187192
1475077289536
1475077289536


Phase 2: Default Arguments (The Heap Trap)
As established, default arguments are evaluated only once, the moment the def statement is read, and stored in the function's __defaults__ attribute.

Case A: Immutable Defaults (Safe)

In [ ]:
def add_score(points, total=0):
    # 'total' starts pointing to the 0 stored in __defaults__
    total = total + points  # new var in stack frame
    # Because ints are immutable, 'total' gets a new sticky note in the stack frame.
    # The 0 inside __defaults__ remains untouched.
    return total


print(add_score(5))  # Returns 5
print(add_score(5))  # Returns 5 (The default is still 0)

5
5


Here is the exact micro-step breakdown:

The total = total + points line (Immutable Reassignment)
When the function is called, the local variable total is created before the code even starts running. Python initializes the parameters.

Initialization: Python creates the local sticky note total. Because you didn't pass an argument, Python writes the address from __defaults__ onto this local sticky note.

The RHS (Right Hand Side): Python evaluates total + points. It looks at the local sticky note total, follows it to the Heap (which currently happens to be the same house __defaults__ points to), grabs the 0, and adds points. A brand new integer object is created in the Heap.

The LHS (Left Hand Side): The = sign triggers. Python takes that exact same local sticky note (total) and crosses out the old address. It writes down the address of the newly calculated integer.

Correction to your statement: The RHS and LHS total are technically the exact same local variable. It is not pulling directly from __defaults__ on the RHS; it is pulling from the local variable, which started out holding the __defaults__ address. The = sign simply updates the address written on that local variable.

The List Case (Mutable Mutation)
Your statement: "in case of list they just point to same memory address having a list object"

This is 100% correct. No notes. Because you do bucket.append(item) instead of using an = sign, Python never crosses out the address on the local bucket sticky note.

__defaults__ points to Address 2000.

The local bucket copies Address 2000.

.append() walks up to Address 2000 and puts something inside.

Since both sticky notes still point to Address 2000, they both see the changed list.

Case B: Mutable Defaults (Dangerous)

In [ ]:
def add_item(item, bucket=[]):
    # 'bucket' points to the list stored in __defaults__
    bucket.append(item)
    # Because lists are mutable, we reached inside the __defaults__ list and altered it.
    # We did NOT move the sticky note.
    return bucket


print(add_item("A"))  # Returns ['A']
print(add_item("B"))  # Returns ['A', 'B'] (The default is forever changed)

['A']
['A', 'B']


In [9]:
add_item.__defaults__[0]

['A', 'B']

Phase 3: Closures and the "Backpack" (Nested Functions)
When an inner function is returned, it creates a Closure (a "backpack") to save the outer variables it needs. How it interacts with that backpack depends entirely on mutability and scope keywords.

Case A: The Outer Variable is Immutable (Read-Only)
By default, Python allows an inner function to read a closure variable, but not reassign it.

In [10]:
def outer_immutable():
    x = 10  # Immutable integer

    def inner():
        # READING works perfectly
        print("Reading x:", x)

    return inner


func = outer_immutable()
func()  # Output: Reading x: 10

Reading x: 10


Case B: The Outer Variable is Immutable (The Crash)
If you try to change that immutable variable, Python panics.

In [ ]:
def outer_crash():
    x = 10

    def inner():
        # Python sees an assignment (=) and assumes 'x' is a brand new LOCAL variable.
        # It forgets about the closure backpack entirely.
        x = x + 1  # CRASH: UnboundLocalError! (Local 'x' referenced before assignment)
        return x

    return inner

Case C: The Outer Variable is Mutable (The Loophole)
If the outer variable is a list or dictionary, you can mutate it without any special keywords. Why? Because you aren't reassigning the sticky note (using =), you are just reaching inside the object in the heap.

In [ ]:
def outer_mutable():
    my_list = []  # Mutable object in the closure backpack

    def inner(item):
        my_list.append(
            item
        )  # We are mutating the heap object, not reassigning the variable!
        return my_list

    return inner


add_to_closure = outer_mutable()
print(add_to_closure(1))  # Output: [1]
print(add_to_closure(2))  # Output: [1, 2]

[1]
[1, 2]


Phase 4: Overriding Scope Restrictions (nonlocal and global)
What if you really need to reassign an immutable variable inside a closure, avoiding the crash in Phase 3, Case B? You use scoping keywords.

Case A: The nonlocal Keyword
nonlocal tells Python: "Do not create a new local sticky note. Go look in the closure backpack and use the sticky note you find there."

In [ ]:
def counter_factory():
    count = 0  # Immutable integer in the outer scope

    def increment():
        nonlocal count  # explicitly grabs 'count' from the closure backpack
        count = count + 1  # Moves the sticky note inside the backpack to a new integer
        return count

    return increment


my_counter = counter_factory()
print(my_counter())  # Output: 1
print(my_counter())  # Output: 2

1
2


In [15]:
print(my_counter.__closure__[0].cell_contents)  # Output: 2

2


In [16]:
print(my_counter())

3


In [17]:
print(my_counter.__closure__[0].cell_contents)  # Output: 3

3


Behind the scenes: In the CPython backend, the closure is stored as a tuple of cell objects. When you use nonlocal, Python specifically updates the memory reference inside that cell object in the heap.

Case B: The global Keyword
global ignores the closure backpack completely. It tells Python: "Go all the way to the top of the file (the module namespace) and use the sticky note there."

In [ ]:
app_state = "Offline"  # Global variable


def system_manager():
    app_state = (
        "Starting"  # This is a local variable, it does NOT change the global one
    )

    def force_online():
        global app_state  # Ignores the "Starting" variable completely
        app_state = "Online"  # Changes the global variable at the top of the file

    return force_online


boot_system = system_manager()
boot_system()
print(app_state)  # Output: "Online"

Online


Summary Cheat Sheet of the Heap Hierarchy
Global Scope: Lives at the module level. Accessible anywhere via global.

Function Definition: Creates the Function Object in the Heap. Default parameters (__defaults__) are locked in right now.

Local Scope (Stack Frame): Created when a function is called (()). Destroyed when it returns.

Closure Scope (__closure__): The backpack. Keeps specific outer local variables alive in the Heap even after the outer Stack Frame dies.

Reassignment vs Mutation: * x = 5 (Reassignment): Creates a new sticky note(local to inner function).
x=x+1 : Fails in inner functions without global or nonlocal.

x.append(5) (Mutation): Modifies the existing Heap object. Works perfectly inside inner functions without any special keywords.

1. x = 5 (Reassignment only)
As you noted, this does not crash. Python simply assumes you want a brand new local sticky note named x. It completely ignores the closure backpack.

Result: It creates a local x inside the inner function. The outer x remains untouched.

In [ ]:
def outer():
    x = 10

    def inner():
        x = 5  # Creates a new local sticky note. No error!
        print("Inner:", x)  # Prints 5

    inner()
    print("Outer:", x)  # Prints 10

In [20]:
outer()

Inner: 5
Outer: 10


2. x = x + 1 (Read + Reassignment)
This is the one that actually crashes with an UnboundLocalError.
Because Python sees the = sign, it decides: "Ah, x must be a local variable for this function." But when it evaluates the right side (x + 1), it realizes the local x doesn't have a value yet. It refuses to look in the closure backpack because the = sign already locked x in as a local variable.

In [ ]:
def outer():
    x = 10

    def inner():
        x = x + 1  # CRASH! Python thinks 'x' is local but it has no value yet.

    inner()

In [24]:
# outer() crashes

Reassignment vs Mutation: > * x = 5 (Reassignment): Creates a new sticky note (local to the inner function). The outer variable is untouched.

x = x + 1 (Read + Reassign): Fails and crashes in inner functions without global or nonlocal.

x.append(5) (Mutation): Modifies the existing Heap object. Works perfectly inside inner functions without special keywords.

In [25]:
x = 10  # Global variable


def my_function():
    print(x)  # Throws UnboundLocalError!
    x = 5  # Because of this assignment, 'x' is local for the WHOLE function

Python runs lines line-by-line, but it compiles the code first.

Before Python executes even the first line of your function, it looks at the entire function block to translate it into bytecode.

During this compilation phase, Python's compiler scans the code and notices: "Ah, there is an x = 5 assignment inside this function, and no global keyword." Because of that discovery, Python instantly flags x as a local variable for the entire scope of that function before execution ever begins.

When the code actually runs line-by-line:

It hits print(x).

Python looks at its internal map and says, "I know x is a local variable."

It checks if the local x has been given a value yet.

Since execution hasn't reached x = 5 yet, the local x is empty, so it throws an UnboundLocalError.

Python chooses this behavior to prevent accidental bugs. If it printed the global x first and then switched x to a local variable on the next line, a single variable name would mean two completely different things in the same function, causing massive confusion.

1. Closures Store Addresses, Not Objects
Just like default arguments, a closure never stores the actual object (like the list [1, 2] or the integer 10).

When Python creates a closure, it puts a "sticky note" (a memory address) inside a special container called a cell.

The cell lives in the closure's backpack.

The sticky note inside the cell points to a house in the Heap.

2. Why Lists (Mutables) Can Be Changed
When you use a method like my_list.append(3) inside an inner function:

Python opens the closure's cell and looks at the sticky note.

It follows that address to the Heap.

It opens the door to the list and puts 3 inside.

The Result: The sticky note inside the cell never had to be crossed out or moved. It still points to the exact same house. This is why it works perfectly without any special keywords.

3. Why Immutables (Integers/Strings) Can Only Be Read
If you just want to read an immutable variable (e.g., print(x)), it works perfectly. Python looks at the sticky note in the cell, goes to the Heap, and reads the value.

The Crash (x = x + 1):
If you try to update an immutable variable, it crashes. But it does not crash because the cell is locked. It crashes because of how Python reads the = sign.

Here is the exact timeline of the crash:

The Parsing Phase (Before the code runs): Python scans the inner function. It sees x = .... Python's strict rule is: "Any variable on the left side of an = sign is officially a local variable for this function."

The Execution Phase: Python actually runs the line x = x + 1.

The RHS (Right Hand Side): Python tries to do x + 1. It asks, "What is x?"

The Error: Because of step 1, Python thinks x is local. It checks the local sticky note for x, but it is completely blank (unassigned). It completely ignores the closure backpack.

Python instantly crashes with: UnboundLocalError: local variable 'x' referenced before assignment.

Summary: The Role of nonlocal
Because Python refuses to look in the closure backpack the moment it sees an = sign, you are stuck only being able to read immutable variables.

When you write nonlocal x, you are telling Python's parser: "When you see x = ... later, do NOT mark it as a local variable. I give you permission to go into the closure backpack, cross out the old sticky note inside the cell, and write down a new memory address."

You are updating the address written on those specific, existing sticky notes.

When you use global or nonlocal, you are not changing the physical object in the Heap (because integers/strings can't be changed). You are simply granting Python permission to cross out the old address on the outer/global sticky note and write a new one.

Here is the exact breakdown of the difference:

1. Without nonlocal / global
When Python sees x = 11, its default behavior is defensive:

"I am not allowed to touch the outer sticky note named x. I will create a brand new, local sticky note named x and write Address 5555 on it."

2. With nonlocal / global
When you explicitly declare nonlocal x or global x, you change Python's behavior:

"The programmer gave me the master key. I will NOT create a local sticky note. I will walk out to the closure backpack (or the global scope), find the original sticky note named x, cross out Address 1000, and write Address 5555 directly on it."

Summary
You are exactly right. You aren't altering the memory object itself; you are just telling Python exactly which sticky note it is allowed to update when it performs the reassignment.

Your intuition is absolutely razor-sharp. You suspected that it does not create a copy, and that multiple closures would end up pointing to the exact same list. You are 100% correct. This is the ultimate "gotcha" in Python. When you combine Mutable Defaults with Closures, you create a scenario where completely separate functions secretly share the exact same house in the Heap.

Here is the complete, step-by-step breakdown of how this happens, missing nothing.

The Scenario: Default Mutable in the Outer Function

In [ ]:
def outer(item, my_list=[]):
    my_list.append(item)

    def inner(new_item):
        my_list.append(new_item)
        return my_list

    return inner

The Step-by-Step Memory Timeline
Step 1: The Definition Phase
Just like we discussed before, Python reads the def outer... line and builds the function.

It creates an empty list [] in the Heap at Address 2000.

It saves Address 2000 inside outer.__defaults__.

Step 2: The First Call -> func1 = outer("A")
Python creates a local sticky note for my_list and copies Address 2000 onto it.

my_list.append("A") goes to Address 2000 and puts "A" inside. The list is now ["A"].

Python builds inner. It creates a closure (backpack). Inside the cell, it puts a sticky note pointing to the local my_list (which is Address 2000).

outer returns inner, which we save as func1.

Result: func1's closure points to Address 2000.

Step 3: The Second Call -> func2 = outer("B")
Python creates a new local sticky note for my_list. Because you didn't pass a list, it grabs the default from __defaults__... which is Address 2000.

my_list.append("B") goes to Address 2000. The list already has "A" in it, so it adds "B". The list is now ["A", "B"].

Python builds a brand new inner function. It creates a new closure cell. Inside the cell, it puts a sticky note pointing to Address 2000.

outer returns this new inner, which we save as func2.

Result: func2's closure also points to Address 2000.

Step 4: The Inner Function Calls
Because both func1 and func2 have sticky notes pointing to the exact same house, calling one affects the other!

Call func1("C"): Goes to Address 2000, adds "C". List is ["A", "B", "C"].

Call func2("D"): Goes to Address 2000, adds "D". List is ["A", "B", "C", "D"].

The Code Proof
Here is the exact code you can run to prove this. We will print the memory addresses of the closures to show they are identical.

In [ ]:
def outer(item, my_list=[]):
    my_list.append(item)

    def inner(new_item):
        my_list.append(new_item)
        return my_list

    return inner


print("--- CREATING CLOSURES ---")
# Create the first inner function
func1 = outer("A")
print(f"func1 closure address: {id(func1.__closure__[0].cell_contents)}")

# Create the second inner function
func2 = outer("B")
print(f"func2 closure address: {id(func2.__closure__[0].cell_contents)}")

print("\n--- EXECUTING CLOSURES ---")
# Watch how calling func1 affects the list that func2 is also looking at
print("Calling func1 with 'C':", func1("C"))
print("Calling func2 with 'D':", func2("D"))

# The ultimate proof: Are they looking at the exact same object?
print("\nAre func1 and func2 sharing the exact same list?")
print(
    func1.__closure__[0].cell_contents is func2.__closure__[0].cell_contents
)  # Output: True

--- CREATING CLOSURES ---
func1 closure address: 1475077339264
func2 closure address: 1475077339264

--- EXECUTING CLOSURES ---
Calling func1 with 'C': ['A', 'B', 'C']
Calling func2 with 'D': ['A', 'B', 'C', 'D']

Are func1 and func2 sharing the exact same list?
True


In [28]:
id(func1) != id(func2)

True

In [29]:
l = func1(5)

In [30]:
l

['A', 'B', 'C', 'D', 5]

In [31]:
func1.__closure__[0].cell_contents

['A', 'B', 'C', 'D', 5]

Summary of the Mechanics
Does it create a copy? No. Closures never copy data. They only copy the memory address (the sticky note).

What happens to the parameters? If the parameter was populated by __defaults__, then every single closure generated by that outer function will hold a sticky note pointing to that one, single default object in the Heap.

The Danger: This creates a "shared state." Every instance of the inner function you create will inadvertently step on the toes of every other instance, because they are all blindly mutating the exact same list.

You mapped out the logic perfectly in your question. You understood that because defaults don't change their address, and closures only copy addresses, they must collide. Spot on!

Let’s break down the exact mechanical proof for each of more three points.

1. Do func1 and func2 have different memory addresses?
Yes, they are completely different function objects. Why? Because in Python, def is an executable statement.
When you call outer() the first time, Python runs the def inner... code and builds a brand new Function Object in the Heap. When you call outer() a second time, it runs def inner... again and builds a second, entirely separate Function Object.

They are two completely different houses in the Heap, but they both happen to have a backpack pointing to the exact same default list.

The Code Proof:

In [ ]:
def outer(item, my_list=[]):
    my_list.append(item)

    def inner(new_item):
        my_list.append(new_item)
        return my_list

    return inner


func1 = outer("A")
func2 = outer("B")

print(f"ID of func1: {id(func1)}")
print(f"ID of func2: {id(func2)} <-- Different function object!")

print(f"\nClosure of func1 points to: {id(func1.__closure__[0].cell_contents)}")
print(
    f"Closure of func2 points to: {id(func2.__closure__[0].cell_contents)} <-- Same list object!"
)

ID of func1: 1475077879088
ID of func2: 1475077879440 <-- Different function object!

Closure of func1 points to: 1475077924608
Closure of func2 points to: 1475077924608 <-- Same list object!


2. What if we pass a 2nd parameter (overriding the default)?
Yes, you break the shared link.

If you explicitly pass a list, Python does not look at __defaults__. It takes the address of the list you passed and writes it on the local my_list sticky note.

When inner is created, its closure backpack grabs that specific sticky note. It will have absolutely nothing to do with the default list.

The Code Proof:

In [ ]:
# func1 uses the default list
func1 = outer("A")

# func3 overrides the default with a custom list!
custom_list = ["X"]
func3 = outer("Y", custom_list)

print(f"func1 closure points to: {id(func1.__closure__[0].cell_contents)}")
print(
    f"func3 closure points to: {id(func3.__closure__[0].cell_contents)} <-- Completely different address!"
)

func3("Z")
print(custom_list)  # Output: ['X', 'Y', 'Z']

func1 closure points to: 1475077924608
func3 closure points to: 1475077339456 <-- Completely different address!
['X', 'Y', 'Z']


In [36]:
id(custom_list) == id(func3.__closure__[0].cell_contents)

True

In [37]:
custom_list

['X', 'Y', 'Z']

In [38]:
func3("W")

['X', 'Y', 'Z', 'W']

In [42]:
custom_list

['X', 'Y', 'Z', 'W']

In [45]:
custom_list is func3.__closure__[0].cell_contents

True

In [46]:
custom_list == func3.__closure__[0].cell_contents

True

Because you provided custom_list, func3 operates safely on its own isolated memory address.

3. What about normal, immutable, or non-default parameters?
They behave exactly like normal local variables. There is zero difference.

In Python, a "parameter" is literally just a local variable that gets its initial sticky note assigned the moment the function is called, rather than being assigned halfway down the function block.

If it is an immutable parameter (like an integer): The closure can read it perfectly. If you try to do param = param + 1 inside the inner function, it will crash with UnboundLocalError unless you use nonlocal param.

If it is a mutable parameter (like a dictionary): The closure can mutate it (param["key"] = "value") perfectly without any special keywords.

Summary of the Golden Rule:
Once the Stack Frame for outer() is created, Python does not care if a variable came from a parameter, from __defaults__, or if you typed x = 10 on line 2. They are all just local sticky notes. The closure backpack treats them all by the exact same rules of Mutability vs. Reassignment.

